# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 1 — PriDe debiasing diagnostic

Reads the scored parquet files written by `stage_1.ipynb` (with PriDe
debiasing) and produces the **letter-position diagnostic**: a check on
whether the model's predictions depend on which MCQ letter the true
answer happened to be assigned to.

The full metrics suite (Brier, calibration, ΔH_c, slices) lives in a
separate notebook; this notebook is focused on the PriDe ablation.

**Schema-aware:** the new `stage_1.ipynb` writes both `prob_raw_{c}` and
`prob_deb_{c}` columns. This notebook detects which kinds are present
and, when both are available, reports diagnostics under both side-by-side
so the PriDe correction can be judged. If only `prob_{c}` is present
(legacy run), it is treated as raw and the comparison panels collapse
gracefully.

**The diagnostic logic (corrected from earlier draft):** under per-row
uniform letter permutation, the right test for letter bias is
**spread across letters**, not deviation from $1/n_Y$. A model that is
informative but unbiased produces constant non-uniform probabilities
across letters; a model that is letter-biased produces varying
probabilities. The summary statistic per (target, kind) is therefore
$$
\text{mean class-spread} = \frac{1}{n_Y} \sum_c \Big[\max_L \bar P(c \mid L) - \min_L \bar P(c \mid L)\Big]
$$
PriDe is "working" if this number shrinks meaningfully from raw to debiased.

**Section overview**

| §  | Output                                                      |
|----|-------------------------------------------------------------|
| 1  | Setup: load all 7 scored parquets, schema-detect prob kinds |
| 2  | Per-letter mean true-class probability (centered)           |
| 3  | Class × letter crosstabs (with marginal column)             |
| 4  | Headline summary: mean / max class-spread, raw vs deb       |


## 1. Setup

Load the per-target scored parquet + metadata sidecar produced by
`stage_1.ipynb`. If a target is missing, surface that loudly rather than
silently skipping it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Model under analysis -------------------------------------------------
#MODEL_NAME = "mistralai/Ministral-3-14B-Instruct-2512"
#MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"
#MODEL_NAME = "Qwen3-4B-Instruct-2507-FT_cumulative_bf16_05_06"
#MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507_05_05"

#MODEL_NAMclE = "Qwen2.5-14B-Instruct-FT_natural_canonical_step07_bf16_test_balanced_05_11"
#MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_balanced_05_11"
MODEL_NAME = "Qwen2.5-14B-Instruct_baseline_test_raw_05_13"

# --- Display knobs --------------------------------------------------------
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# --- I/O ------------------------------------------------------------------
SCORED_DIR  = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/"
)
METRICS_DIR = Path(
    f"/content/drive/MyDrive/NLP_project/data/"
    f"stage_1_output_{MODEL_NAME}/phase1_metrics/"
)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# --- Targets to analyze ---------------------------------------------------
TARGETS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]


In [ ]:
def detect_prob_kinds(scored: pd.DataFrame, classes: list) -> list:
    """Return the list of probability-column kinds available in the parquet.

    Stage 1 with PriDe writes prob_raw_{c} and prob_deb_{c} (and prob_{c}
    aliased to debiased). The pre-PriDe pipeline only wrote prob_{c}.

    Returns one of:
      ["raw", "deb"]   -- new schema (PriDe-enabled run)
      ["raw"]          -- legacy schema (only prob_{c} present; treat as raw)
    """
    has_raw = all(f"prob_raw_{c}" in scored.columns for c in classes)
    has_deb = all(f"prob_deb_{c}" in scored.columns for c in classes)
    if has_raw and has_deb:
        return ["raw", "deb"]
    return ["raw"]


def load_target(target: str) -> dict:
    """Load scored parquet + metadata for one target."""
    pq_path  = SCORED_DIR / f"phase1_scored_{target}.parquet"
    meta_path = SCORED_DIR / f"phase1_scored_{target}.meta.json"

    if not pq_path.exists():
        raise FileNotFoundError(f"Missing scored parquet for {target}: {pq_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata for {target}: {meta_path}")

    scored = pd.read_parquet(pq_path)
    with open(meta_path) as f:
        meta = json.load(f)

    classes = meta["class_labels"]
    prob_kinds = detect_prob_kinds(scored, classes)

    # Legacy fallback: if only prob_{c} is present, alias it to prob_raw_{c}
    # so all downstream code uses the same column-name pattern. This is only
    # valid when apply_pride was False during scoring (i.e. genuinely raw);
    # under PriDe, prob_{c} aliases prob_deb_{c}, but in that case both
    # prob_raw_{c} and prob_deb_{c} also exist so this branch isn't taken.
    needs_legacy_alias = (
        prob_kinds == ["raw"]
        and not all(f"prob_raw_{c}" in scored.columns for c in classes)
    )
    if needs_legacy_alias:
        for c in classes:
            scored[f"prob_raw_{c}"] = scored[f"prob_{c}"]

    return {
        "scored":        scored,
        "meta":          meta,
        "classes":       classes,
        "prob_kinds":    prob_kinds,
        "applied_pride": meta.get("apply_pride", False),
    }


def prob_cols(classes: list, kind: str = "deb") -> list:
    """Column names for probabilities of `kind` in {'raw', 'deb'}."""
    return [f"prob_{kind}_{c}" for c in classes]


# Try loading everything up front. Missing targets become explicit warnings.
runs = {}
for t in TARGETS:
    try:
        runs[t] = load_target(t)
        n = len(runs[t]["scored"])
        k = len(runs[t]["classes"])
        kinds = "+".join(runs[t]["prob_kinds"])
        pride_tag = "PriDe" if runs[t]["applied_pride"] else "raw-only"
        print(f"  {t:<20} loaded: {n:,} rows, {k} classes, "
              f"kinds=[{kinds}] ({pride_tag})")
    except FileNotFoundError as e:
        warnings.warn(f"Skipping {t}: {e}")

print(f"\nLoaded {len(runs)} / {len(TARGETS)} targets.")

# Default kind for headline diagnostics. If any loaded run lacks `deb`, we
# fall back to `raw` so legacy parquets still produce results.
ALL_HAVE_DEB = all("deb" in r["prob_kinds"] for r in runs.values())
DEFAULT_KIND = "deb" if ALL_HAVE_DEB else "raw"
print(f"\nDefault prob kind for headline: {DEFAULT_KIND!r}")


In [ ]:
# Quick inspection helpers
print("Loaded targets:", list(runs.keys()))


## 2. Letter-position diagnostic

For each row $i$ with letter map $\sigma_i: \{A, B, \ldots\} \to \{1, \ldots, n_Y\}$
and each letter $L$, define
$$\bar P(L) = \frac{1}{N} \sum_i P_{\text{LLM}}\!\left(Y = \sigma_i(L) \mid X_{-Y,i}\right).$$

Under no letter-position bias, $\bar P(L)$ is constant in $L$ — equal to
the model's overall mean true-class probability, **not** $1/n_Y$. The
$1/n_Y$ baseline only applies at chance. The right diagnostic for
letter bias is therefore **spread across letters**, not deviation from
$1/n_Y$.

We also compute the **class × letter** crosstab:
$$\bar P(c \mid L) = \frac{1}{|\{i : \sigma_i(L) = c\}|}
  \sum_{i: \sigma_i(L) = c} P_{\text{LLM}}(Y = c \mid X_{-Y, i}),$$
which separates a global letter prior from class-conditional ones.
Under uniform per-row permutation, the cell-size-weighted row mean of
this crosstab equals the unconditional marginal of $\text{prob\_kind\_c}$;
we assert this internally as a sanity check.


In [ ]:
def letter_position_summary(scored: pd.DataFrame, classes: list,
                            kind: str = "deb") -> pd.DataFrame:
    """Per-letter mean predicted probability of the true class, bar P(L).

    Under no letter bias, mean_p is approximately constant across letters
    (at the model's overall mean true-class probability, NOT at 1/n_Y).
    The bias diagnostic is therefore spread across letters, returned in
    the `centered_dev` column for plotting.
    """
    letter_maps = scored["letter_map_json"].apply(json.loads).tolist()
    letters = sorted({L for lm in letter_maps for L in lm})
    n_classes = len(classes)

    rows_by_letter = {L: [] for L in letters}
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    probs = scored[prob_cols(classes, kind)].to_numpy()

    for i, lm in enumerate(letter_maps):
        for L, c in lm.items():
            rows_by_letter[L].append(probs[i, cls_to_idx[c]])

    out = pd.DataFrame({
        "letter":  letters,
        "n_rows":  [len(rows_by_letter[L]) for L in letters],
        "mean_p":  [float(np.mean(rows_by_letter[L])) for L in letters],
        "uniform": 1.0 / n_classes,
    })
    out["centered_dev"] = out["mean_p"] - out["mean_p"].mean()
    return out


def class_letter_crosstab(scored: pd.DataFrame, classes: list,
                          kind: str = "deb") -> pd.DataFrame:
    """Mean probability matrix indexed by (class, letter).

    Cell (c, L) = mean of P(Y=c | X) over rows where letter L was assigned
    to class c. Under uniform per-row permutation, the cell-size-weighted
    row mean equals the unconditional marginal of prob_{kind}_{c}. We
    warn if the discrepancy exceeds 1e-6 (sampling-design bug).
    """
    letter_maps = scored["letter_map_json"].apply(json.loads).tolist()
    letters = sorted({L for lm in letter_maps for L in lm})
    cls_to_idx = {c: i for i, c in enumerate(classes)}
    probs = scored[prob_cols(classes, kind)].to_numpy()

    bucket = {(c, L): [] for c in classes for L in letters}
    for i, lm in enumerate(letter_maps):
        for L, c in lm.items():
            bucket[(c, L)].append(probs[i, cls_to_idx[c]])

    mat = pd.DataFrame(
        [[np.mean(bucket[(c, L)]) if bucket[(c, L)] else np.nan
          for L in letters]
         for c in classes],
        index=classes,
        columns=letters,
    )

    # Sanity: cell-size-weighted average across letters should equal the
    # unconditional marginal of prob_{kind}_{c}.
    cell_sizes = pd.DataFrame(
        [[len(bucket[(c, L)]) for L in letters] for c in classes],
        index=classes, columns=letters,
    )
    cell_sums = cell_sizes.sum(axis=1).replace(0, np.nan)
    weighted_avg = (mat.fillna(0) * cell_sizes).sum(axis=1) / cell_sums
    marginal = scored[prob_cols(classes, kind)].mean()
    marginal.index = classes
    discrepancy = float((weighted_avg - marginal).abs().max())
    if discrepancy > 1e-6:
        warnings.warn(
            f"class_letter_crosstab: weighted-cell-mean vs unconditional "
            f"marginal differ by {discrepancy:.2e}; check per-row permutation."
        )
    return mat


def position_bias_score(crosstab: pd.DataFrame) -> dict:
    """Per-class spread across letters. Lower = better debiased.

    Returns:
      mean_spread : average over classes of (max_L p - min_L p)
      max_spread  : worst class
      by_class    : per-class spread series
    """
    spreads = {}
    for c in crosstab.index:
        row = crosstab.loc[c].dropna()
        if len(row) > 1:
            spreads[c] = float(row.max() - row.min())
    spreads_arr = (np.array(list(spreads.values())) if spreads
                   else np.array([0.0]))
    return {
        "mean_spread": float(spreads_arr.mean()),
        "max_spread":  float(spreads_arr.max()),
        "by_class":    pd.Series(spreads),
    }


In [ ]:
# Compute diagnostics for both raw and deb (when available).

letter_summaries = {}    # (target, kind) -> DataFrame
letter_crosstabs = {}    # (target, kind) -> DataFrame
bias_scores      = {}    # (target, kind) -> dict from position_bias_score
for t, run in runs.items():
    s = run["scored"]
    cls = run["classes"]
    for kind in run["prob_kinds"]:
        summary  = letter_position_summary(s, cls, kind)
        crosstab = class_letter_crosstab(s, cls, kind)
        letter_summaries[(t, kind)] = summary
        letter_crosstabs[(t, kind)] = crosstab
        bias_scores[(t, kind)]      = position_bias_score(crosstab)

print(f"Computed diagnostics for {len(letter_summaries)} (target, kind) pairs.")


## 3. Headline: mean / max class-spread, raw vs debiased

The single number to report per target. Reduction is computed as
$1 - \text{mean\_spread\_deb} / \text{mean\_spread\_raw}$ and is
expressed as a percentage. Higher = PriDe removed more bias.


In [ ]:
summary_rows = []
for t in runs:
    row = {"target": t, "n_classes": len(runs[t]["classes"])}
    for kind in ("raw", "deb"):
        if (t, kind) in bias_scores:
            row[f"mean_spread_{kind}"] = bias_scores[(t, kind)]["mean_spread"]
            row[f"max_spread_{kind}"]  = bias_scores[(t, kind)]["max_spread"]
        else:
            row[f"mean_spread_{kind}"] = np.nan
            row[f"max_spread_{kind}"]  = np.nan
    raw_mean = row.get("mean_spread_raw", np.nan)
    deb_mean = row.get("mean_spread_deb", np.nan)
    if np.isfinite(raw_mean) and np.isfinite(deb_mean) and raw_mean > 0:
        row["reduction_pct"] = 100.0 * (1.0 - deb_mean / raw_mean)
    else:
        row["reduction_pct"] = np.nan
    summary_rows.append(row)

bias_summary = pd.DataFrame(summary_rows).set_index("target")
print("PriDe debiasing summary (mean class-spread across letters)")
print("=" * 72)
print("Lower spread = less letter bias. PriDe should reduce raw -> deb.")
print()
print(bias_summary.to_string(float_format=lambda x: f"{x:.3f}"))

bias_summary_path = METRICS_DIR / "bias_summary.csv"
bias_summary.to_csv(bias_summary_path)
print(f"\nSaved -> {bias_summary_path}")


In [ ]:
# --- PriDe summary bar plot ------------------------------------------------
# Reads the `bias_summary` DataFrame built above and draws a two-panel
# figure: per-target mean/max spread (raw vs deb) on top, reduction-%
# on the bottom. Sorted by reduction_pct descending so the targets where
# PriDe worked best appear leftmost.

# Sort and drop targets with no reduction info (e.g. legacy raw-only runs)
plot_df = (
    bias_summary
    .dropna(subset=["reduction_pct"])
    .sort_values("reduction_pct", ascending=False)
)

x = np.arange(len(plot_df))
width = 0.36

RAW_COLOR = "#c44e52"
DEB_COLOR = "#55a868"
RAW_LIGHT = "#e9b6b3"
DEB_LIGHT = "#bcd9b8"

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(11, 7),
    gridspec_kw={"height_ratios": [3, 1]},
    sharex=True,
)

# --- Top panel: mean spread (foreground) over max spread (background) ---
# Background bars (max spread) — slightly wider, lighter, behind.
ax1.bar(x - width / 2, plot_df["max_spread_raw"], width=width * 1.05,
        color=RAW_LIGHT, edgecolor="none",
        label="max spread (raw)", zorder=1)
ax1.bar(x + width / 2, plot_df["max_spread_deb"], width=width * 1.05,
        color=DEB_LIGHT, edgecolor="none",
        label="max spread (deb)", zorder=1)

# Foreground bars (mean spread)
b1 = ax1.bar(x - width / 2, plot_df["mean_spread_raw"], width=width,
             color=RAW_COLOR, edgecolor="white", linewidth=0.5,
             label="mean spread (raw)", zorder=2)
b2 = ax1.bar(x + width / 2, plot_df["mean_spread_deb"], width=width,
             color=DEB_COLOR, edgecolor="white", linewidth=0.5,
             label="mean spread (deb)", zorder=2)

# Annotate each mean bar with its value
for bar in b1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
             f"{h:.3f}", ha="center", va="bottom",
             fontsize=8, color=RAW_COLOR, fontweight="bold")
for bar in b2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
             f"{h:.3f}", ha="center", va="bottom",
             fontsize=8, color=DEB_COLOR, fontweight="bold")

ax1.set_ylabel(
    r"class-spread across letters: $\max_L \bar P(c|L) - \min_L \bar P(c|L)$",
    fontsize=10,
)
ax1.set_title(
    "PriDe debiasing: per-target letter-bias reduction",
    fontsize=12, pad=10,
)
ymax = max(plot_df["max_spread_raw"].max(),
           plot_df["max_spread_deb"].max()) * 1.15
ax1.set_ylim(0, ymax)
ax1.legend(loc="upper right", frameon=True, fontsize=9, ncol=2,
           framealpha=0.95)
ax1.grid(axis="y", linestyle=":", alpha=0.4)
ax1.spines[["top", "right"]].set_visible(False)

# --- Bottom panel: reduction percentage, color-graded ---
def _reduction_color(pct: float) -> str:
    if pct >= 60:
        return "#2d8659"
    if pct >= 40:
        return "#55a868"
    if pct >= 25:
        return "#a3c486"
    return "#d4a85a"  # amber for weak reduction (<25%)


bar_colors = [_reduction_color(r) for r in plot_df["reduction_pct"]]
b3 = ax2.bar(x, plot_df["reduction_pct"], color=bar_colors,
             edgecolor="white", linewidth=0.5)
for bar, r in zip(b3, plot_df["reduction_pct"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, r + 1.5,
             f"{r:.1f}%", ha="center", va="bottom",
             fontsize=9, fontweight="bold")

ax2.axhline(50, color="grey", linestyle=":", linewidth=0.8, alpha=0.6)
ax2.set_ylabel("PriDe reduction (%)", fontsize=10)
ax2.set_xticks(x)
labels = [
    f"{t}\n(n={int(plot_df.loc[t, 'n_classes'])})"
    for t in plot_df.index
]
ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylim(0, 100)
ax2.grid(axis="y", linestyle=":", alpha=0.4)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(METRICS_DIR / "fig0_pride_summary.png",
            dpi=180, bbox_inches="tight")
plt.savefig(METRICS_DIR / "fig0_pride_summary.pdf",
            bbox_inches="tight")
plt.show()

## 5. Class × letter crosstabs (per target)

Each row is a class; each column a letter. Cell $(c, L) =$ mean
$P(c \mid X)$ over rows where letter $L$ was assigned to class $c$. Two
summary columns are appended:

- **Mean**: unweighted mean across the letter columns. Gives equal voice
  to each letter slot, so it is rotation-symmetric.
- **Marginal**: unconditional mean of $\text{prob\_kind\_c}$ across all
  rows. Every row contributes once, regardless of the letter its class
  landed at.

Under perfectly uniform per-row permutation these are equal. Their gap
$|\text{Mean} - \text{Marginal}|$ measures cell-size imbalance plus any
residual structure that survives rotation.

Each panel title reports two summary numbers:

- **max gap**: $\max_c |\text{Mean}(c) - \text{Marginal}(c)|$. The
  worst-class deviation. This is the design diagnostic — answers "is
  any class's cell distribution off-uniform?" Used as the primary flag.
- **pop-weighted gap**: $\sum_c \pi_c \, |\text{Mean}(c) - \text{Marginal}(c)|$
  where $\pi_c$ are ACS population shares (recovered from
  `hajek_weight`). The fidelity diagnostic — answers "does letter bias
  contaminate the *population-weighted* marginal we'd otherwise report?"
  Often very small even when max gap is non-trivial, because minority
  classes (where imbalance tends to bite) get downweighted.

A row that is **constant across letters** indicates no letter bias for
that class. A row that varies indicates the model's prediction for that
class depends on which letter the class was assigned. PriDe should
flatten rows from raw to debiased.


In [ ]:
try:
    from IPython.display import display as _display
except ImportError:
    def _display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)


def _population_class_weights(scored: pd.DataFrame, target_col: str,
                              classes: list) -> pd.Series:
    """Recover ACS population shares pi_c from Hajek weights.

    Sampling design: hajek_weight_i = pi_{c_i} * N / n_{c_i}, so summing
    within class c gives pi_c * N. Normalizing recovers pi_c.
    """
    if "hajek_weight" not in scored.columns:
        # Fallback: equal weights (degrades gracefully on legacy parquets).
        return pd.Series(1.0 / len(classes), index=classes)
    sums = scored.groupby(target_col, observed=True)["hajek_weight"].sum()
    pi = sums / sums.sum()
    return pi.reindex(classes).fillna(0.0)


for t, run in runs.items():
    kinds = run["prob_kinds"]
    fig, axes = plt.subplots(
        1, len(kinds),
        figsize=(max(8, 7 * len(kinds)), 5),
    )
    if len(kinds) == 1:
        axes = [axes]

    fig.suptitle(
        f"{t} - Class x Letter probability  "
        f"(rows where letter L was assigned to class c)",
        fontsize=12, y=1.02,
    )

    pi = _population_class_weights(run["scored"], t, run["classes"])

    for ax, kind in zip(axes, kinds):
        mat = letter_crosstabs[(t, kind)].copy()

        # Mean (unweighted average across letter columns) is computed only
        # to derive the gap diagnostics; it is NOT shown in the table.
        # Marginal is the unconditional mean of prob_{kind}_{c} across all
        # rows. Under uniform per-row permutation these are equal.
        letter_cols = list(mat.columns)
        mean_col = mat[letter_cols].mean(axis=1)
        marginal_col = pd.Series(
            run["scored"][prob_cols(run["classes"], kind)].mean().values,
            index=run["classes"],
        )

        mat["Marginal"] = marginal_col

        # Two summary statistics on the gap:
        #   - max: design diagnostic (catches outlier classes)
        #   - pop-weighted: fidelity diagnostic (matches the headline marginal)
        gap_per_class = (mean_col - marginal_col).abs()
        max_gap     = float(gap_per_class.max())
        pop_w_gap   = float((pi * gap_per_class).sum())

        score = bias_scores[(t, kind)]
        print(f"\n[{t} | {kind}]  mean_spread={score['mean_spread']:.3f}, "
              f"max_spread={score['max_spread']:.3f}, "
              f"max gap={max_gap:.4f}, pop-weighted gap={pop_w_gap:.4f}")
        _display(mat.round(3))

        # Adaptive vmax so small-mass classes stay readable on imbalanced
        # targets (e.g. RACE_ETHNICITY where Chinese/Japanese ~ 0).
        vmax = max(0.2, float(np.nanmax(mat.values)))
        sns.heatmap(
            mat, annot=True, fmt=".3f", cmap="Blues",
            ax=ax, vmin=0, vmax=vmax,
        )
        ax.set_title(
            f"kind: {kind}   max gap={max_gap:.4f}   "
            f"pop-w gap={pop_w_gap:.4f}",
            fontsize=11,
        )
        ax.set_ylabel("class")
        ax.set_xlabel("letter / marginal")

    plt.tight_layout()
    plt.savefig(
        METRICS_DIR / f"fig0b_crosstab_{t}.png",
        dpi=150, bbox_inches="tight",
    )
    plt.show()